## Step 0 — Install the repaired repository (Kaggle only)

Kaggle sessions do not start with the repository present, so the first cell clones the repair branch into `/kaggle/working/graph-fraud-benchmark` and verifies what landed. It deletes any previous clone first, so a fresh session always starts from the repair branch rather than stale session files.

The verification compares the cloned `src/` tree against a pinned commit. It cannot simply compare the clone's HEAD to a SHA stored in this notebook — that notebook is itself committed, so the commit it names is always one behind HEAD. Checking that `src/` is byte-identical to the pinned commit is the equivalent guarantee, and it is what the rest of the notebook actually executes.

This cell is Kaggle environment bootstrap only. It is not part of the benchmark package and never touches `/kaggle/benchmarks`. It makes no model calls.


In [ ]:
# --- Kaggle setup: install the exact repaired repository -------------------
# Kaggle environment bootstrap only. This cell is not part of the benchmark
# package, and it never reads or writes /kaggle/benchmarks.
#
# Any previous clone is deleted so that every fresh Kaggle session starts from
# the repair branch rather than from whatever the previous session left behind.
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/yourfavCHEFP/graph-fraud-benchmark.git"
REPO_BRANCH = "fix/gemma-4-31b-benchmark-repair"
REPO_DIR = Path("/kaggle/working/graph-fraud-benchmark")

# The commit whose src/ tree this notebook was validated against. The branch
# head sits one commit ahead of it -- the commit carrying this notebook -- so
# HEAD cannot be compared for equality against a SHA stored inside that very
# notebook. Instead HEAD is verified by proving its src/ tree is byte-identical
# to this commit, which is what the notebook actually executes.
EXPECTED_REPAIR_COMMIT = "f64b95dd3bc9072b25f12b1de260c8d85a3d9f34"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

clone = subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
    timeout=300,
)
print(clone.stdout.strip())
print(clone.stderr.strip())
if clone.returncode != 0:
    raise RuntimeError(f"git clone failed ({clone.returncode}): {clone.stderr.strip()}")

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("KAGGLE REPO COMMIT:", commit)
print("EXPECTED REPAIR COMMIT (src/ baseline):", EXPECTED_REPAIR_COMMIT)

# A depth-1 clone has no history, so fetch the baseline commit to diff against.
fetch = subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", EXPECTED_REPAIR_COMMIT],
    capture_output=True, text=True, timeout=300,
)
if fetch.returncode != 0:
    raise RuntimeError(
        f"could not fetch the verified repair commit {EXPECTED_REPAIR_COMMIT}: "
        f"{fetch.stderr.strip()}"
    )

src_diff = subprocess.run(
    ["git", "-C", str(REPO_DIR), "diff", "--quiet", EXPECTED_REPAIR_COMMIT, commit, "--", "src/"]
)
if src_diff.returncode != 0:
    raise RuntimeError(
        f"cloned src/ differs from the verified repair commit {EXPECTED_REPAIR_COMMIT}; "
        f"HEAD is {commit}. Refusing to run against unverified source."
    )

# The modules whose APIs this notebook calls directly.
missing = [
    name for name in (
        "src/graph_fraud_benchmark/responses.py",
        "src/graph_fraud_benchmark/sample_subgraphs.py",
        "src/graph_fraud_benchmark/gnn_predictions.py",
    )
    if not (REPO_DIR / name).is_file()
]
if missing:
    raise RuntimeError(f"cloned repository is missing required modules: {missing}")

print("CORRECT REPAIRED REPOSITORY IS INSTALLED.")

# Graph Fraud Reasoning Benchmark

This benchmark compares language-model decisions with a GraphSAGE fraud detector on the same held-out IEEE-CIS transactions. Each model receives a small, label-free description of the seed transaction and its linked neighbors.

It evaluates two questions:

- **Fraud classification:** is the seed transaction fraudulent?
- **Fraud-neighbor proxy:** did the model identify another linked transaction that is independently labeled fraudulent? IEEE-CIS has no verified ring-leader labels, so this is a proxy, not ring-leader detection.

The notebook checks graph artifacts, samples balanced seeds from the chronological GNN test period, serializes prompts, evaluates models, and compares errors with the GNN baseline. Mock mode checks pipeline wiring only; use real Kaggle model runs for benchmark results.

In [ ]:
# --- Bootstrap: locate the repo clone and put its package on the path -------
# Kaggle clones this repository to /kaggle/working/graph-fraud-benchmark while
# the notebook's working directory is /kaggle/working. A cwd-relative guess
# therefore resolves to /kaggle/working, which contains no src/ at all, and
# every package import below fails with ModuleNotFoundError -- which looks like
# a missing file but is really a wrong path. This cell finds the clone
# properly and imports it src-layout style: `graph_fraud_benchmark.x`, with
# <repo>/src on sys.path. It never reads /kaggle/benchmarks and makes no model
# calls.
import importlib
import os
import sys
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

_PACKAGE_MARKER = Path("src") / "graph_fraud_benchmark" / "__init__.py"


def _is_repo(path: Path) -> bool:
    return (path / _PACKAGE_MARKER).is_file()


def _repo_candidates():
    """Places a clone can legitimately live, most specific first."""
    env = os.environ.get("GRAPH_FRAUD_BENCHMARK_SOURCE_DIR")
    if env:
        yield Path(env).expanduser()
    yield KAGGLE_WORKING_ROOT / "graph-fraud-benchmark"
    yield Path.cwd()
    # One level under the working directory, and under the cwd itself, so a
    # clone is found whether it lands in /kaggle/working/<name> or wherever
    # the kernel happens to be started from.
    seen_roots = []
    for root in (KAGGLE_WORKING_ROOT, Path.cwd()):
        resolved = root.resolve()
        if resolved.is_dir() and resolved not in seen_roots:
            seen_roots.append(resolved)
            for child in sorted(resolved.iterdir()):
                # "benchmarks" is Kaggle's own framework directory, never a
                # source tree -- excluded so it can never be mistaken for one.
                if child.is_dir() and child.name != "benchmarks":
                    yield child
    yield Path.cwd().parent


def _find_repo_root() -> Path:
    for candidate in _repo_candidates():
        if candidate.is_dir() and _is_repo(candidate):
            return candidate.resolve()
    searched = ", ".join(str(c) for c in _repo_candidates())
    raise FileNotFoundError(
        "Could not find the graph-fraud-benchmark clone. Searched: "
        f"{searched}. Set GRAPH_FRAUD_BENCHMARK_SOURCE_DIR to the clone root."
    )


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"

# The package's modules import each other relatively (`from .responses import`),
# so `graph_fraud_benchmark.x` is the correct form once SRC_DIR is on the path.
# REPO_ROOT is deliberately NOT added as well: that would also make
# `src.graph_fraud_benchmark` importable as a second, independent copy of every
# module, and two copies of a module do not share state. A stale `src.` import
# should fail loudly rather than quietly bind to a duplicate.
while str(SRC_DIR) in sys.path:
    sys.path.remove(str(SRC_DIR))
sys.path.insert(0, str(SRC_DIR))

import graph_fraud_benchmark
from graph_fraud_benchmark.models import get_model_lineup

# --- Prove the repaired module is loaded from THIS clone -------------------
# Compared against the resolved src/ directory rather than REPO_ROOT, so a
# clone reached through a symlinked path still verifies correctly while a
# genuinely foreign copy of the package still fails.
responses_module = importlib.import_module("graph_fraud_benchmark.responses")
RESPONSES_PATH = Path(responses_module.__file__).resolve()
if SRC_DIR.resolve() not in RESPONSES_PATH.parents:
    raise RuntimeError(
        f"graph_fraud_benchmark.responses resolved to {RESPONSES_PATH}, which is "
        f"outside the clone's src/ at {SRC_DIR.resolve()}. Another copy of the "
        "package is shadowing it."
    )

# --- Verify the lineup every later pinning check assumes -------------------
# ModelSpec's fields are display_name / kaggle_model_id / role. There is no
# .name and no .model_id -- reading those raises AttributeError.
EXPECTED_LINEUP = [
    ("Gemini 2.5 Pro", "google/gemini-2.5-pro"),
    ("Gemini 2.5 Flash", "google/gemini-2.5-flash"),
    ("Gemma 4 31B", "google/gemma-4-31b"),
]
MODEL_LINEUP = get_model_lineup()
if [(m.display_name, m.kaggle_model_id) for m in MODEL_LINEUP] != EXPECTED_LINEUP:
    raise ValueError(
        "MODEL_LINEUP does not match the expected order. The pinning checks in "
        "Step 6 and Step 8 assume Gemma 4 31B sits at index 2. Got: "
        f"{[(m.display_name, m.kaggle_model_id) for m in MODEL_LINEUP]}"
    )

# --- Paths, all absolute under the clone ------------------------------------
# Kaggle input datasets are read-only, but the clone under /kaggle/working is
# writable, so generated data and results live beside the source.
OUTPUT_ROOT = REPO_ROOT
DATA_DIR = OUTPUT_ROOT / "data"
RESULTS_DIR = OUTPUT_ROOT / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

raw_override = os.environ.get("IEEE_CIS_INPUT_DIR")
local_raw = REPO_ROOT / "data" / "raw"
KAGGLE_RAW_DATA_DIR = KAGGLE_INPUT_ROOT / "competitions" / "ieee-fraud-detection"
transaction_candidates = (
    list(KAGGLE_INPUT_ROOT.rglob("train_transaction.csv"))
    if KAGGLE_INPUT_ROOT.exists()
    else []
)
RAW_DATA_DIR = (
    Path(raw_override)
    if raw_override
    else KAGGLE_RAW_DATA_DIR
    if (KAGGLE_RAW_DATA_DIR / "train_transaction.csv").is_file()
    else transaction_candidates[0].parent
    if transaction_candidates
    else local_raw
)
TRANSACTIONS_CSV = RAW_DATA_DIR / "train_transaction.csv"
IDENTITY_CSV = RAW_DATA_DIR / "train_identity.csv"

local_gnn = REPO_ROOT / "data" / "gnn"
gnn_override = os.environ.get("GRAPH_FRAUD_GNN_ARTIFACT_DIR")
gnn_candidates = (
    list(KAGGLE_INPUT_ROOT.rglob("fraud_graph_ready.pt"))
    if KAGGLE_INPUT_ROOT.exists()
    else []
)
if gnn_override:
    GNN_ARTIFACT_DIR = Path(gnn_override)
elif (local_gnn / "fraud_graph_ready.pt").is_file():
    GNN_ARTIFACT_DIR = local_gnn
elif gnn_candidates:
    GNN_ARTIFACT_DIR = gnn_candidates[0].parent
else:
    GNN_ARTIFACT_DIR = local_gnn

print("repo root            :", REPO_ROOT)
print("package path entry   :", sys.path[0])
print("responses.py loaded  :", RESPONSES_PATH)
print("output root          :", OUTPUT_ROOT)
print("transactions CSV     :", TRANSACTIONS_CSV, "exists:", TRANSACTIONS_CSV.is_file())
print("identity CSV exists  :", IDENTITY_CSV.is_file())
print("GNN artifact root    :", GNN_ARTIFACT_DIR)
print("GNN graph exists     :", (GNN_ARTIFACT_DIR / "fraud_graph_ready.pt").is_file())
print("pinned model         :", f"MODEL_LINEUP[2] = {MODEL_LINEUP[2].display_name} "
      f"({MODEL_LINEUP[2].kaggle_model_id})")

## Step 1 — Sample subgraphs

Calls `sample_subgraphs.run()` directly rather than shelling out, so this cell's variables (the entity columns, the real per-entity-type hub caps confirmed against graph-fraud-ai's own production model) are exactly the tested code from the repo, not a reimplementation.

In [ ]:
import pandas as pd

from graph_fraud_benchmark.sample_subgraphs import run as sample_subgraphs_run, HUB_DEGREE_CAPS

prediction_path = DATA_DIR / "gnn" / "test_predictions.csv"
precomputed_predictions = GNN_ARTIFACT_DIR / "test_predictions.csv"
if precomputed_predictions.is_file():
    # Use the locally validated export; Kaggle's task kernel may not include PyTorch.
    gnn_predictions = pd.read_csv(precomputed_predictions)
    print(f"using precomputed GNN scores from {precomputed_predictions}")
else:
    # Local fallback: compute scores when the PyTorch inference dependencies exist.
    # On Kaggle this branch imports torch, so attach a dataset carrying
    # test_predictions.csv (or GRAPH_FRAUD_GNN_ARTIFACT_DIR) to stay on the
    # branch above.
    from graph_fraud_benchmark.gnn_predictions import run as gnn_predictions_run

    gnn_predictions_run(
        graph_path=GNN_ARTIFACT_DIR / "fraud_graph_ready.pt",
        checkpoint_path=GNN_ARTIFACT_DIR / "graphsage_improved.pt",
        node_features_path=GNN_ARTIFACT_DIR / "node_features.parquet",
        transactions_csv=TRANSACTIONS_CSV,
        out_csv=prediction_path,
    )
    gnn_predictions = pd.read_csv(prediction_path)

sample_subgraphs_run(
    transactions_csv=TRANSACTIONS_CSV,
    identity_csv=IDENTITY_CSV if IDENTITY_CSV.exists() else None,
    out_dir=DATA_DIR,
    n_seeds_per_class=50,
    hops=2,
    max_neighbors_per_entity=4,
    max_subgraph_size=12,
    hub_degree_caps=HUB_DEGREE_CAPS,
    seed=42,
    eligible_seed_ids=set(gnn_predictions["TransactionID"].astype(int)),
    gnn_predictions=dict(
        zip(gnn_predictions["TransactionID"].astype(int), gnn_predictions["gnn_pred"])
    ),
)

## Step 2 — Serialize into prompts

Label never passed in — see `serialize.py`'s own docstring for why that's structurally guaranteed, not just a promise.

In [ ]:
from graph_fraud_benchmark.serialize import run as serialize_run

serialize_run(
    subgraphs_dir=DATA_DIR / "subgraphs",
    out_jsonl=DATA_DIR / "serialized_subgraphs.jsonl",
)

## Step 3 — Model lineup

`kaggle_model_id` values are placeholders — confirm against Kaggle's live model catalog (the model picker in this notebook, or https://kaggle.com/benchmarks) before the real run.

In [ ]:
# MODEL_LINEUP was already resolved and verified by the bootstrap cell. This
# cell just displays it with explicit indices, because the pinning in Steps 6
# and 8 refers to lineup positions by number.
#
# ModelSpec's fields are display_name / kaggle_model_id / role. There is no
# .name and no .model_id on this object.
for _index, _spec in enumerate(MODEL_LINEUP):
    print(f"MODEL_LINEUP[{_index}] = {_spec.display_name:20s} {_spec.kaggle_model_id:25s} {_spec.role}")

_target = MODEL_LINEUP[2]
assert (_target.display_name, _target.kaggle_model_id) == ("Gemma 4 31B", "google/gemma-4-31b")
print(f"\npinned target: MODEL_LINEUP[2] = {_target.display_name} ({_target.kaggle_model_id})")

## Step 4 — Model adapter

`tasks.py`'s functions expect any `model` object with a `.generate(prompt) -> response` method where `response.text` is a string — that contract was written generically on purpose. `KaggleModelAdapter` below is where Kaggle's actual model-calling API gets plugged in.

**VERIFY ME:** the body of `KaggleModelAdapter.generate()` is the one piece in this whole repo I can't test from here — Kaggle Benchmarks' exact model invocation call isn't something I can confirm without hands-on access to a live notebook. Wire in the real call once you're inside Kaggle; everything downstream (tasks.py, assertions.py, analysis.py) only needs `.generate(prompt).text` to keep working, so nothing else in the pipeline should need to change.

`MockModel` is a stand-in that needs no Kaggle access at all, so the *rest* of this notebook's wiring can be dry-run and proven correct independently of that one unverified piece.

In [ ]:
import random
from dataclasses import dataclass


@dataclass
class ModelResponse:
    text: str


class KaggleModelAdapter:
    """Adapter for Kaggle Benchmarks' documented LLM API."""

    def __init__(self, kaggle_model_id: str):
        import kaggle_benchmarks as kbench

        self._llm = kbench.llms[kaggle_model_id]

    def generate(self, prompt: str) -> ModelResponse:
        return ModelResponse(text=self._llm.prompt(prompt))


class MockModel:
    """Deterministic stand-in for local pipeline checks only."""

    def __init__(self, seed: int = 0):
        self._rng = random.Random(seed)

    def generate(self, prompt: str) -> ModelResponse:
        if "FRAUD or LEGIT" in prompt:
            return ModelResponse(text=self._rng.choice(["FRAUD", "LEGIT", "NOT FRAUD"]))
        ids_in_prompt = [int(tok) for tok in prompt.split() if tok.isdigit() and len(tok) >= 3]
        seed_marker = "THE TRANSACTION IN QUESTION] Transaction "
        seed_id = None
        for line in prompt.splitlines():
            if seed_marker in line:
                seed_id = int(line.split(seed_marker, 1)[1].split(":", 1)[0])
                break
        candidates = [txn_id for txn_id in ids_in_prompt if txn_id != seed_id]
        if candidates and self._rng.random() > 0.3:
            return ModelResponse(
                text=f"Transaction {self._rng.choice(candidates)} looks responsible."
            )
        return ModelResponse(text="NONE")


# Mock outputs validate plumbing, not model quality or Kaggle submission status.
USE_MOCK = True

## Step 5 — Mock responses (pipeline check only)

With `USE_MOCK = True` this loops over every model x every sample x both tasks using the deterministic `MockModel`, and writes `results/mock_raw_responses.csv` in exactly the shape `analysis.py` expects: `model_name, task, seed_txn_id, raw_response`. Only the raw text goes in this file — no parsing happens here, so there is exactly one place (tasks.py / assertions.py) that interprets what a model's answer means.

This is plumbing validation, not a benchmark result: the mock model is not any of the models in the lineup. Real responses are produced and checkpointed by the run cell in Step 6, which writes `model3_pilot_raw_responses.csv` or `raw_responses.csv` depending on the mode.

In [ ]:
import json
import pandas as pd

from graph_fraud_benchmark.tasks import build_classification_prompt, build_ring_root_prompt

with open(DATA_DIR / "serialized_subgraphs.jsonl") as f:
    samples = [json.loads(line) for line in f]

rows = []
if USE_MOCK:
    # Local dry run only. Real requests run through Kaggle's task API below.
    for model_index, model_spec in enumerate(MODEL_LINEUP):
        model = MockModel(seed=model_index)
        for sample in samples:
            for task_name, prompt_builder in (
                ("classify_transaction", build_classification_prompt),
                ("identify_ring_root", build_ring_root_prompt),
            ):
                response = model.generate(prompt_builder(sample["prompt"]))
                rows.append({
                    "model_name": model_spec.display_name,
                    "task": task_name,
                    "seed_txn_id": sample["seed_txn_id"],
                    "raw_response": response.text,
                })

    results_df = pd.DataFrame(rows)
    results_path = RESULTS_DIR / "mock_raw_responses.csv"
    results_df.to_csv(results_path, index=False)
    print(f"wrote {len(results_df)} mock responses to {results_path}")
    results_df.head()
else:
    print("Live responses will be generated and recorded by the Kaggle task in Step 6.")

## Step 6 — Run the Kaggle benchmark

Kaggle Benchmarks runs decorated tasks with an `llm` argument and `llm.prompt()`. Set `USE_MOCK = False` to run the configured model through the task below.

This cell is **pinned to one model**: `MODEL_LINEUP[2]` (`google/gemma-4-31b`), in both pilot and full mode. The pin is enforced by explicit checks that raise rather than substitute, so no code path can quietly fall back to `MODEL_LINEUP[0]` (Gemini 2.5 Pro) or any other entry. The previous version of this cell sliced `MODEL_LINEUP[:1]` for the pilot, which ran Gemini while the pilot cache was named as though it belonged to Gemma.

Each call is validated before it is stored: the returned object is unwrapped, the text is stripped, and an empty result is retried **once** with a fresh chat. If the second attempt is also empty the run stops with a diagnostic and **nothing is checkpointed for that key** — an HTTP 200 with no content is never recorded as a completed result. Answers are written atomically to disk under the key `(model_name, task, seed_txn_id)` as each one arrives, and on resume a key counts as done only if its cached response is real, non-empty text. Blank cached rows are left on disk but stay eligible to be re-requested.

Pilot mode runs 5 samples × 2 tasks = **10 non-empty Gemma responses** into the model-specific `results/model3_pilot_raw_responses.csv`. Full mode runs 100 samples × 2 tasks = **200 non-empty responses** into `results/raw_responses.csv`, reusing the pilot's answers. A cache holding rows from a different model raises instead of being merged.

In [ ]:
if USE_MOCK:
    print("Mock responses were written in Step 5; they are not Kaggle benchmark results.")
else:
    import kaggle_benchmarks as kbench

    from graph_fraud_benchmark.assertions import score_classification, score_ring_root
    from graph_fraud_benchmark.responses import (
        TASKS,
        EmptyModelResponseError,
        completed_keys_from_frame,
        completion_key,
        is_complete_response,
        read_responses_csv,
        request_response,
        validate_response_run,
        write_responses_csv,
    )
    from graph_fraud_benchmark.tasks import _classify_logic, _ring_root_logic

    # --- What this run is allowed to touch ---------------------------------
    # The repair target is ONE model: lineup entry #3, Gemma 4 31B. Both
    # modes select it by explicit index. Nothing here falls back to
    # MODEL_LINEUP[0] -- the previous pilot sliced MODEL_LINEUP[:1], which is
    # why the "Gemma" pilot was really a second Gemini run. The checks below
    # raise rather than substitute, so a lineup edit can never silently
    # retarget the benchmark.
    TARGET_MODEL_INDEX = 2
    EXPECTED_KAGGLE_MODEL_ID = "google/gemma-4-31b"
    EXPECTED_MODEL_NAME = "Gemma 4 31B"
    if not 0 <= TARGET_MODEL_INDEX < len(MODEL_LINEUP):
        raise ValueError(f"TARGET_MODEL_INDEX {TARGET_MODEL_INDEX} is out of range")
    TARGET_MODEL = MODEL_LINEUP[TARGET_MODEL_INDEX]
    if TARGET_MODEL.kaggle_model_id != EXPECTED_KAGGLE_MODEL_ID:
        raise ValueError(
            f"MODEL_LINEUP[{TARGET_MODEL_INDEX}] is {TARGET_MODEL.kaggle_model_id!r}, "
            f"but this run is pinned to {EXPECTED_KAGGLE_MODEL_ID!r}. Refusing to "
            "run against a different model."
        )
    if TARGET_MODEL.display_name != EXPECTED_MODEL_NAME:
        raise ValueError(
            f"MODEL_LINEUP[{TARGET_MODEL_INDEX}] is named "
            f"{TARGET_MODEL.display_name!r}, expected {EXPECTED_MODEL_NAME!r}"
        )
    print(
        f"target model: MODEL_LINEUP[{TARGET_MODEL_INDEX}] = "
        f"{TARGET_MODEL.display_name} ({TARGET_MODEL.kaggle_model_id})"
    )

    RUN_MODE = "pilot"  # change to "full" only after Step 8 verifies the pilot
    if RUN_MODE not in {"pilot", "full"}:
        raise ValueError("RUN_MODE must be 'pilot' or 'full'")

    MAX_OUTPUT_TOKENS = 128
    PILOT_SAMPLE_COUNT = 5
    EXPECTED_SAMPLES = {"pilot": 5, "full": 100}
    EXPECTED_RESPONSES = {"pilot": 10, "full": 200}

    samples_to_run = samples[:PILOT_SAMPLE_COUNT] if RUN_MODE == "pilot" else samples
    if len(samples_to_run) != EXPECTED_SAMPLES[RUN_MODE]:
        raise ValueError(
            f"{RUN_MODE} mode needs {EXPECTED_SAMPLES[RUN_MODE]} samples, "
            f"got {len(samples_to_run)}"
        )
    # The same single model in BOTH modes. Deliberately not MODEL_LINEUP[:1].
    models_to_run = [TARGET_MODEL]

    # The pilot cache is named for the model it belongs to, so a Gemini pilot
    # and a model-#3 pilot can never land in -- or be read from -- one file.
    PILOT_RESULTS_PATH = (
        RESULTS_DIR / f"model{TARGET_MODEL_INDEX + 1}_pilot_raw_responses.csv"
    )
    FULL_RESULTS_PATH = RESULTS_DIR / "raw_responses.csv"
    REAL_RESULTS_PATH = (
        PILOT_RESULTS_PATH if RUN_MODE == "pilot" else FULL_RESULTS_PATH
    )
    print(f"{RUN_MODE} results path: {REAL_RESULTS_PATH}")

    labels = pd.read_csv(DATA_DIR / "labels.csv").set_index("seed_txn_id")
    raw_labels = pd.read_csv(TRANSACTIONS_CSV, usecols=["TransactionID", "isFraud"])
    node_is_fraud = dict(zip(raw_labels["TransactionID"], raw_labels["isFraud"]))

    # --- Resume -------------------------------------------------------------
    # A key counts as DONE only if its cached response is real, non-empty
    # text. Blank/NaN cached rows are left on disk but excluded from the
    # completed map, so they stay eligible to be re-requested rather than
    # being mistaken for finished work. expected_model raises on any row
    # recorded for a different model, so caches cannot be mixed.
    cache_paths = (
        [PILOT_RESULTS_PATH]
        if RUN_MODE == "pilot"
        else [PILOT_RESULTS_PATH, FULL_RESULTS_PATH]
    )
    results_by_key: dict = {}
    cached_incomplete: list = []
    for cache_path in cache_paths:
        if not cache_path.is_file():
            continue
        cached_frame = read_responses_csv(cache_path)
        results_by_key.update(
            completed_keys_from_frame(
                cached_frame, expected_model=TARGET_MODEL.display_name
            )
        )
        if cache_path == REAL_RESULTS_PATH:
            cached_incomplete = (
                cached_frame[~cached_frame["response_complete"]].to_dict("records")
            )

    def save_responses():
        """Atomically checkpoint this run's answers.

        Completed rows plus any still-unanswered rows already in this run's own
        file, so a half-finished run keeps its place. A blank row that has
        since been answered is dropped in favour of the real response, and the
        temp-file replace means an interrupted write can't truncate the file.
        """
        rows = list(results_by_key.values())
        rows.extend(
            row
            for row in cached_incomplete
            if completion_key(
                row["model_name"], row["task"], row["seed_txn_id"]
            )
            not in results_by_key
        )
        write_responses_csv(rows, REAL_RESULTS_PATH)

    def ask(llm, prompt, task_name, seed_id):
        """Send one prompt, validate the reply, and retry at most once.

        Each attempt opens its own chat, so the retry is a genuinely new
        request rather than a second look at the same empty envelope. Raises
        rather than returning "" -- an empty answer must never reach the CSV.
        """

        def call(text):
            with kbench.chats.new(f"{task_name} {seed_id}"):
                return llm.prompt(
                    text,
                    extra_api_params={
                        "extra_body": {"max_tokens": MAX_OUTPUT_TOKENS}
                    },
                )

        return request_response(prompt, call, task_name, seed_id)

    def get_or_ask(llm, key, task_name, prompt, seed_id):
        """Return the response for `key`, resuming it or requesting it once.

        Only a validated, non-empty response is ever stored under `key`, and it
        is checkpointed the moment it is real.
        """
        if key in results_by_key:
            return results_by_key[key]["raw_response"]
        response = ask(llm, prompt, task_name, seed_id)
        results_by_key[key] = {
            "model_name": key[0],
            "task": key[1],
            "seed_txn_id": key[2],
            "raw_response": response,
        }
        save_responses()
        return response

    print(
        f"Starting {RUN_MODE}: {len(samples_to_run)} samples x "
        f"{len(models_to_run)} model ({TARGET_MODEL.display_name}); "
        f"resuming {len(results_by_key)} non-empty responses"
    )

    @kbench.task(
        name=f"graph_fraud_benchmark_{RUN_MODE}"
        f"_model{TARGET_MODEL_INDEX + 1}_tok{MAX_OUTPUT_TOKENS}"
    )
    def graph_fraud_benchmark(llm, model_name: str) -> float:
        if model_name != TARGET_MODEL.display_name:
            raise ValueError(
                f"refusing to score {model_name!r}: this run is pinned to "
                f"{TARGET_MODEL.display_name!r}"
            )
        classification_scores = []
        ring_root_scores = []

        for sample in samples_to_run:
            seed_id = int(sample["seed_txn_id"])
            true_label = int(labels.loc[seed_id, "label"])

            classification_response = get_or_ask(
                llm,
                completion_key(model_name, "classify_transaction", seed_id),
                "classify_transaction",
                build_classification_prompt(sample["prompt"]),
                seed_id,
            )
            parsed = _classify_logic(classification_response)
            classification_score = score_classification(
                parsed.predicted_fraud, parsed.parse_confidence, true_label
            )
            if classification_score.correct is not None:
                classification_scores.append(classification_score.correct)

            ring_response = get_or_ask(
                llm,
                completion_key(model_name, "identify_ring_root", seed_id),
                "identify_ring_root",
                build_ring_root_prompt(sample["prompt"]),
                seed_id,
            )
            # Guarded explicitly: a blank parses to no transaction ID, which
            # reads as a deliberate "NONE" and would score a legit seed as a
            # correct answer. get_or_ask should have raised already, so this is
            # the backstop that keeps silence from counting as a reply.
            if not is_complete_response(ring_response):
                raise EmptyModelResponseError(
                    f"identify_ring_root for seed {seed_id} is blank after resume"
                )
            predicted_id = _ring_root_logic(
                ring_response, sample["valid_txn_ids"], seed_id
            )
            ring_score = score_ring_root(predicted_id, true_label, node_is_fraud)
            if ring_score.correct is not None:
                ring_root_scores.append(ring_score.correct)

            print(
                f"[{RUN_MODE}] {model_name}: {len(results_by_key)} non-empty "
                f"responses through seed {seed_id}"
            )

        classification_accuracy = (
            sum(classification_scores) / len(classification_scores)
            if classification_scores
            else 0.0
        )
        ring_root_accuracy = (
            sum(ring_root_scores) / len(ring_root_scores)
            if ring_root_scores
            else 0.0
        )
        print(model_name, {
            "classification_accuracy": classification_accuracy,
            "fraud_neighbor_proxy_accuracy": ring_root_accuracy,
        })
        return (classification_accuracy + ring_root_accuracy) / 2

    try:
        for model_spec in models_to_run:
            if model_spec.kaggle_model_id != EXPECTED_KAGGLE_MODEL_ID:
                raise ValueError(
                    f"refusing to call {model_spec.kaggle_model_id!r}: this run is "
                    f"pinned to {EXPECTED_KAGGLE_MODEL_ID!r}"
                )
            graph_fraud_benchmark.run(
                llm=kbench.llms[model_spec.kaggle_model_id],
                model_name=model_spec.display_name,
            )
    except EmptyModelResponseError as exc:
        raise RuntimeError(
            f"[{RUN_MODE} stopped] {exc} Every completed response is preserved in "
            f"{REAL_RESULTS_PATH}; re-running this cell resumes from there."
        ) from exc

    save_responses()

    # Final gate. This counts NON-EMPTY responses -- a CSV with the right
    # number of rows and nothing in them fails here, which is the whole point.
    summary = validate_response_run(
        read_responses_csv(REAL_RESULTS_PATH),
        expected_model=TARGET_MODEL.display_name,
        expected_samples=[int(s["seed_txn_id"]) for s in samples_to_run],
        expected_tasks=TASKS,
    )
    if summary["non_empty_responses"] != EXPECTED_RESPONSES[RUN_MODE]:
        raise ValueError(
            f"{RUN_MODE} produced {summary['non_empty_responses']} non-empty "
            f"responses, expected {EXPECTED_RESPONSES[RUN_MODE]}"
        )
    print(
        f"[{RUN_MODE}] complete: {summary['non_empty_responses']} non-empty "
        f"{summary['model_name']} responses across {summary['samples']} samples "
        f"-> {REAL_RESULTS_PATH}"
    )
    if RUN_MODE == "pilot":
        print("Next: run Step 8 and check the pilot leaderboard, then set RUN_MODE = 'full'.")

## Step 7 — GNN predictions

The prediction exporter runs in Step 1 using the corrected GraphSAGE checkpoint and matching graph artifacts. It verifies transaction-ID ordering, labels, feature names, and train-only normalization before writing predictions. The selected benchmark seeds come only from the chronological held-out test period, and `labels.csv` carries each seed's GNN probability.

In [ ]:
# The matching graph, corrected production checkpoint, and node-feature mapping
# must be attached locally under data/gnn or as a Kaggle Dataset whose path is
# set in GRAPH_FRAUD_GNN_ARTIFACT_DIR. The exporter verifies their feature and
# normalization contracts before producing any predictions.
print("GNN test predictions:", DATA_DIR / "gnn" / "test_predictions.csv")

## Step 8 — Analyze: build the leaderboard

In [ ]:
# --- Step 8: analyze the run that actually happened ----------------------
from graph_fraud_benchmark.analysis import run as analysis_run
from graph_fraud_benchmark.responses import read_responses_csv, validate_response_run

# Pinned to the same model the run cell is pinned to. A lineup edit that moves
# Gemma off index 2 must fail loudly here rather than quietly analyze a
# different model's results.
TARGET_MODEL_INDEX = 2
if MODEL_LINEUP[TARGET_MODEL_INDEX].kaggle_model_id != "google/gemma-4-31b":
    raise ValueError(
        f"Step 8 is pinned to google/gemma-4-31b, but MODEL_LINEUP[{TARGET_MODEL_INDEX}] "
        f"is {MODEL_LINEUP[TARGET_MODEL_INDEX].kaggle_model_id!r}"
    )
TARGET_MODEL_NAME = MODEL_LINEUP[TARGET_MODEL_INDEX].display_name

if USE_MOCK:
    analysis_results_path = RESULTS_DIR / "mock_raw_responses.csv"
    analysis_output_path = RESULTS_DIR / "mock_leaderboard_export.json"
else:
    # No fallback. If Step 6 never ran in this kernel, say so instead of
    # quietly analyzing whichever raw_responses.csv happens to be lying around
    # -- a stale or foreign cache would otherwise be scored and reported as
    # this run's leaderboard.
    if "RUN_MODE" not in globals() or "REAL_RESULTS_PATH" not in globals():
        raise RuntimeError(
            "Step 6 has not run in this kernel (RUN_MODE / REAL_RESULTS_PATH are "
            "undefined), so there is no known results file to analyze. Re-run Step 6 "
            "with USE_MOCK = False. Refusing to guess at a file name."
        )

    if RUN_MODE == "pilot":
        # Model-specific by construction: the pilot cache for lineup entry #3.
        # Never the Gemini pilot, never a scaffold file, never the full-run file.
        analysis_results_path = RESULTS_DIR / "model3_pilot_raw_responses.csv"
        analysis_output_path = RESULTS_DIR / "pilot_leaderboard_export.json"
    else:
        analysis_results_path = RESULTS_DIR / "raw_responses.csv"
        analysis_output_path = RESULTS_DIR / "leaderboard_export.json"

    if analysis_results_path != REAL_RESULTS_PATH:
        raise RuntimeError(
            f"Step 8 resolved {analysis_results_path} but Step 6 wrote "
            f"{REAL_RESULTS_PATH}; refusing to analyze a file this run did not produce."
        )
    if not analysis_results_path.is_file():
        raise FileNotFoundError(
            f"{RUN_MODE} mode found no results at {analysis_results_path}. Re-run Step 6."
        )

    expected_samples = [
        int(s["seed_txn_id"]) for s in (samples[:5] if RUN_MODE == "pilot" else samples)
    ]
    # Counts NON-EMPTY responses for the pinned model: 10 for the pilot, and
    # 200 (100 samples x 2 tasks) for the full run. Blank raw_response cells do
    # not count, so 200 blank rows is explicitly NOT a completed run.
    summary = validate_response_run(
        read_responses_csv(analysis_results_path),
        expected_model=TARGET_MODEL_NAME,
        expected_samples=expected_samples,
    )
    print(
        f"[step 8] {RUN_MODE}: {summary['non_empty_responses']} non-empty "
        f"{summary['model_name']} responses across {summary['samples']} samples "
        f"({summary['by_task']})"
    )

analysis_run(
    results_csv=analysis_results_path,
    labels_csv=DATA_DIR / "labels.csv",
    serialized_jsonl=DATA_DIR / "serialized_subgraphs.jsonl",
    raw_transactions_csv=TRANSACTIONS_CSV,
    out_path=analysis_output_path,
    gnn_threshold=0.198,
)

## Results and next steps

Step 6 starts in **pilot** mode, pinned to `MODEL_LINEUP[2]` (Gemma 4 31B): five held-out samples, two tasks each, ten LLM calls. Each validated answer is checkpointed to `results/model3_pilot_raw_responses.csv` as it arrives, so progress survives an interruption and re-running resumes rather than re-paying. The pilot cache is named for the model it belongs to and cannot be mixed with another model's.

The required progression — do not skip a step:

1. Run Step 6 in pilot mode and confirm it reports **10 non-empty Gemma responses**.
2. Run Step 8. It re-reads `model3_pilot_raw_responses.csv`, verifies the same 10 non-empty responses for `Gemma 4 31B`, and writes the pilot leaderboard. Review it and confirm both task scores look sensible.
3. Only then change `RUN_MODE` in Step 6 from `"pilot"` to `"full"`, re-run Step 6, and re-run Step 8. Full mode evaluates 100 samples for the same pinned model and two tasks (200 LLM calls), and Step 8 will refuse to proceed until `results/raw_responses.csv` holds **200 non-empty Gemma responses**. A file with 200 blank rows does not count as a completed run.
4. Treat only non-mock responses as benchmark results. Describe the second task as the fraud-neighbor proxy, not verified ring-leader detection.
5. After checking the full report, write the observed results and limitations in `writeup/devto_submission.md` before building or publishing the Kaggle task.